# Train YOLOv8s - Bien So Xe Viet Nam (Google Colab GPU)

**Yeu cau**: Runtime > Change runtime type > **T4 GPU**

**Thoi gian uoc tinh**: ~1-2 gio (100 epochs, yolov8s, GPU T4)

**Ket qua**: mAP50 mong doi > 99%, mAP50-95 > 88%

In [ ]:
# Kiem tra GPU
import torch
print(f'CUDA: {torch.cuda.is_available()}')
print(f'GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE"}')
!nvidia-smi

In [ ]:
# Cai dat thu vien
!pip install -q ultralytics roboflow

In [ ]:
# Tai dataset tu Roboflow (can API key)
# Dang ky tai https://roboflow.com de lay API key mien phi

from roboflow import Roboflow

API_KEY = "YOUR_ROBOFLOW_API_KEY"  # <-- thay bang API key cua ban

rf = Roboflow(api_key=API_KEY)
project = rf.workspace("vietnam-license").project("vietnam-license-plate-hjswj")
version = project.version(2)
dataset = version.download("yolov8")
print(f"Dataset tai ve: {dataset.location}")

In [ ]:
# HOAC: Upload dataset tu may tinh (neu khong co API key)
# 1. Zip folder dataset/ tren may tinh
# 2. Upload len Colab bang:

# from google.colab import files
# uploaded = files.upload()  # chon file dataset.zip
# !unzip dataset.zip -d dataset/

In [ ]:
# Kiem tra cau truc dataset
import os
dataset_path = dataset.location  # hoac dat tay: dataset_path = '/content/dataset'

for split in ['train', 'valid', 'test']:
    img_dir = os.path.join(dataset_path, split, 'images')
    if os.path.exists(img_dir):
        count = len(os.listdir(img_dir))
        print(f'{split}: {count} anh')

In [ ]:
from ultralytics import YOLO
import yaml, os

# Cap nhat duong dan trong data.yaml cho Colab
yaml_path = os.path.join(dataset_path, 'data.yaml')
with open(yaml_path) as f:
    data = yaml.safe_load(f)

data['train'] = os.path.join(dataset_path, 'train', 'images')
data['val']   = os.path.join(dataset_path, 'valid', 'images')
data['test']  = os.path.join(dataset_path, 'test',  'images')

with open(yaml_path, 'w') as f:
    yaml.dump(data, f)

print('data.yaml da cap nhat:')
print(yaml.dump(data))

In [ ]:
# TRAIN - YOLOv8s, 100 epochs, GPU
model = YOLO('yolov8s.pt')  # yolov8s chinh xac hon yolov8n

results = model.train(
    data=yaml_path,
    epochs=100,
    imgsz=640,
    batch=32,             # GPU T4 co the chiu batch 32
    device=0,             # GPU
    patience=20,
    optimizer='AdamW',
    lr0=0.001,
    cos_lr=True,
    warmup_epochs=3,
    # Augmentation
    hsv_h=0.02,
    hsv_s=0.7,
    hsv_v=0.5,
    degrees=15.0,
    translate=0.1,
    scale=0.5,
    shear=2.0,
    perspective=0.0005,
    fliplr=0.0,           # bien so khong flip
    mosaic=1.0,
    mixup=0.15,
    copy_paste=0.1,
    erasing=0.4,
    # Luu
    save=True,
    save_period=10,
    project='runs',
    name='bien_so_v2_gpu',
    exist_ok=True,
    plots=True,
)

print('Train xong!')
print(f'mAP50: {results.results_dict["metrics/mAP50(B)"]:.4f}')
print(f'mAP50-95: {results.results_dict["metrics/mAP50-95(B)"]:.4f}')

In [ ]:
# Xem bieu do ket qua
from IPython.display import Image
Image('runs/detect/bien_so_v2_gpu/results.png')

In [ ]:
# Validate tren tap test
best_model = YOLO('runs/detect/bien_so_v2_gpu/weights/best.pt')
metrics = best_model.val(data=yaml_path, split='test')
print(f'Test mAP50:    {metrics.box.map50:.4f}')
print(f'Test mAP50-95: {metrics.box.map:.4f}')
print(f'Precision:     {metrics.box.mp:.4f}')
print(f'Recall:        {metrics.box.mr:.4f}')

In [ ]:
# Tai model ve may tinh
from google.colab import files
files.download('runs/detect/bien_so_v2_gpu/weights/best.pt')
print('Da tai: best.pt')
print('Dat file vao: runs/detect/runs/bien_so_v2_gpu/weights/best.pt')
print('Sau do chay: python demo.py --model runs/detect/runs/bien_so_v2_gpu/weights/best.pt')